# Vision Model Fine-Tuning for a Narrow Image Task

**Goal:** adapt a pre-trained vision transformer to a narrow, domain-specific image
classification task, covering dataset preparation, fine-tuning strategies, evaluation
metrics, and the practical constraints every practitioner needs to know.

---

## The Core Idea

Pre-trained vision models (ViT, Swin, CLIP visual encoder, etc.) have already learned
rich visual features from millions of images. Fine-tuning re-uses those features and
specialises them for your task with far less data and compute than training from scratch.

## What "Narrow Task" Means

A narrow task is one where the domain shift from the pre-training distribution is large:
- medical images (X-rays, histology, dermoscopy)
- industrial inspection (surface defects, PCB anomalies, weld cracks)
- satellite or aerial imagery
- specialised retail items

Narrow tasks share two challenges:
- **small datasets** (hundreds to low thousands of images, not millions)
- **tight precision requirements** (false positives or false negatives have real costs)

## What This Notebook Covers

1. Choosing a pre-trained vision backbone
2. Dataset preparation: loading, augmentation, balance, pre-processing
3. Three fine-tuning strategies compared
4. Training recipe (ViT-base with LoRA adapter)
5. Metrics that matter for narrow tasks
6. Error analysis workflow
7. Practical constraints: VRAM, resolution, data volume, overfitting


## 1. Environment Setup

In [ ]:
!pip install -q torch torchvision transformers peft datasets timm scikit-learn matplotlib seaborn pillow

In [ ]:
import io
import json
import math
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageDraw, ImageFilter
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "synthetic_images"
ARTIFACT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

BASE_MODEL = "google/vit-base-patch16-224"
RUN_TRAINING = False
RUN_INFERENCE_EVAL = False

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifacts:    {ARTIFACT_DIR}")
print(f"Base model:   {BASE_MODEL}")
print(f"Training:     {RUN_TRAINING}")
print(f"Inference:    {RUN_INFERENCE_EVAL}")

## 2. The Narrow Task: Surface Defect Classification

We will simulate an **industrial surface defect classification** task with three classes:

| Class | Description | Real-world analogy |
|---|---|---|
| `no_defect` | Clean surface | Acceptable product |
| `scratch` | Linear surface mark | Machining scratch |
| `crack` | Branching fracture pattern | Structural failure risk |

This is a natural narrow-task scenario because:
- the images look nothing like ImageNet
- visual differences between classes can be subtle
- class imbalance is common (defects are rare in production)
- false negatives have higher cost than false positives

### Dataset Scale

We will generate **300 synthetic images** (100 per class) to keep the notebook
self-contained. In production, this would be 500 - 5000 labelled images per class,
which is typical for inspection workflows.


## 3. Generate Synthetic Dataset

In [ ]:
import torch

def _make_base(size=224, seed=0):
    rng = random.Random(seed)
    img = Image.new("RGB", (size, size))
    draw = ImageDraw.Draw(img)
    # textured metallic-like background
    for _ in range(3000):
        x, y = rng.randint(0, size), rng.randint(0, size)
        v = rng.randint(130, 200)
        draw.point((x, y), fill=(v, v, v - rng.randint(0, 20)))
    img = img.filter(ImageFilter.GaussianBlur(radius=0.8))
    return img


def make_no_defect(idx):
    img = _make_base(seed=idx)
    return img


def make_scratch(idx):
    img = _make_base(seed=idx + 10000)
    draw = ImageDraw.Draw(img)
    rng = random.Random(idx)
    n_lines = rng.randint(1, 3)
    for _ in range(n_lines):
        x0 = rng.randint(20, 180)
        y0 = rng.randint(20, 100)
        x1 = x0 + rng.randint(-30, 30)
        y1 = y0 + rng.randint(60, 140)
        w = rng.choice([1, 2])
        c = rng.randint(20, 80)
        draw.line([(x0, y0), (x1, y1)], fill=(c, c, c), width=w)
    return img


def make_crack(idx):
    img = _make_base(seed=idx + 20000)
    draw = ImageDraw.Draw(img)
    rng = random.Random(idx)
    # branching zigzag crack
    x = rng.randint(40, 180)
    y = rng.randint(10, 60)
    for _ in range(rng.randint(20, 40)):
        dx = rng.randint(-8, 8)
        dy = rng.randint(3, 10)
        x2, y2 = x + dx, y + dy
        c = rng.randint(10, 60)
        draw.line([(x, y), (x2, y2)], fill=(c, c, c), width=2)
        if rng.random() < 0.3:  # branch
            bx2 = x2 + rng.randint(-12, 12)
            by2 = y2 + rng.randint(5, 15)
            draw.line([(x2, y2), (bx2, by2)], fill=(c, c, c), width=1)
        x, y = x2, y2
        if y > 200:
            break
    return img


CLASSES = ["no_defect", "scratch", "crack"]
N_PER_CLASS = 100

records = []
for cls in CLASSES:
    cls_dir = DATA_DIR / cls
    cls_dir.mkdir(exist_ok=True)
    make_fn = {"no_defect": make_no_defect, "scratch": make_scratch, "crack": make_crack}[cls]
    for i in range(N_PER_CLASS):
        img = make_fn(i)
        path = cls_dir / f"{cls}_{i:03d}.png"
        img.save(path)
    records.extend([{"path": str(DATA_DIR / cls / f"{cls}_{i:03d}.png"), "label": cls} for i in range(N_PER_CLASS)])

df = pd.DataFrame(records)
print(f"Total images:   {len(df)}")
print(f"Classes:        {sorted(df['label'].unique())}")
print(f"Class counts:")
print(df["label"].value_counts().to_string())
print(f"\nSaved to: {DATA_DIR}")

In [ ]:
# Visualise one sample from each class
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, cls in zip(axes, CLASSES):
    sample_path = DATA_DIR / cls / f"{cls}_000.png"
    img = Image.open(sample_path)
    ax.imshow(img)
    ax.set_title(cls.replace("_", "\n"), fontsize=11, fontweight="bold")
    ax.axis("off")
plt.suptitle("Synthetic Surface Defect Samples (224×224)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Dataset Preparation

### 4.1 Train / Validation / Test Split

For narrow tasks the split strategy matters more than for large-scale tasks:

| Concern | Recommendation |
|---|---|
| Small total size | Keep test set intact and do not tune on it at all |
| Class imbalance | Stratify by class at every split stage |
| Temporal or domain structure | Split by acquisition date / scanner if available |

A common split for narrow tasks is **60% train, 20% val, 20% test**.

### 4.2 Class Imbalance

Real inspection datasets are heavily imbalanced — defects may account for 1-10% of
images. Handling options:

| Strategy | When to use | Trade-off |
|---|---|---|
| Weighted loss | Always a good baseline | Still trains on all images |
| Oversampling (minority up-sampling) | Small minority class | Risk of overfitting to duplicated images |
| Undersampling (majority) | Majority class is very large | Wastes data |
| Augment the minority class | When augmentation is semantically valid | Requires careful augmentation design |


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"])

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = split_df["label"].value_counts()
    print(f"{split_name:<8} total={len(split_df):>4}   " + "   ".join(f"{c}: {counts[c]}" for c in CLASSES))

label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}
print(f"\nLabel map: {label2id}")

In [ ]:
# Class weight computation for the weighted loss function
from collections import Counter

class_counts = Counter(train_df["label"])
total = len(train_df)
class_weights = [total / (len(CLASSES) * class_counts[c]) for c in CLASSES]
print("Class weights (used in loss if classes are imbalanced):")
for cls, w in zip(CLASSES, class_weights):
    print(f"  {cls:<14} {w:.4f}")

## 5. Augmentation Strategy

### Why Augmentation Is Critical for Narrow Tasks

With only hundreds of images, the model will see each training image many times per
epoch. Without augmentation it will memorize the training set and fail to generalise.

### What to Augment and What Not to

| Augmentation | Narrow task consideration |
|---|---|
| **Horizontal / vertical flip** | Valid for defect inspection. Not valid if orientation encodes class (e.g., digits, text OCR). |
| **Random rotation** | Valid when orientation is arbitrary. May not be valid for satellite imagery with fixed north. |
| **Colour jitter** | Valid when lighting varies. Be conservative — preserve colour as a feature if it matters. |
| **Random crop / resized crop** | Standard. Ensures the model is not fixed to the image centre. |
| **Gaussian blur / noise** | Valid for robustness to imaging noise. Keep sigma small. |
| **Elastic deformation** | Very useful for medical images. Keep magnitude low. |
| **Cutout / random erase** | Effective regularisation. Avoid if the defect is small. |

### Train vs. Inference Pre-processing

These two pipelines should differ:

| Stage | Transform |
|---|---|
| Training | Random crop, flips, colour jitter, resize to model input |
| Validation / inference | Centre crop, resize only — no random transforms |


In [ ]:
import torchvision.transforms as T

IMAGE_SIZE = 224  # standard ViT input size

train_transform = T.Compose([
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomGrayscale(p=0.05),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize(int(IMAGE_SIZE * 1.14)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def denorm(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# Visualise augmented samples
from torch.utils.data import DataLoader, Dataset


class DefectDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label2id[row["label"]]


aug_ds = DefectDataset(train_df[train_df["label"] == "scratch"], transform=train_transform)
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for ax, idx in zip(axes, range(6)):
    t, _ = aug_ds[idx]
    ax.imshow(denorm(t))
    ax.axis("off")
    ax.set_title(f"Aug {idx + 1}", fontsize=9)
plt.suptitle('Six augmented "scratch" samples from the same image set', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Fine-Tuning Strategies

There are three main strategies for adapting a pre-trained vision model.

```
Pre-trained ViT-Base
  Patch embedding → [Block 0 ... Block 11] → [CLS] → Linear head

Strategy A – Linear probe:        freeze all   → only fine-tune head
Strategy B – Partial fine-tuning: freeze early → fine-tune last N blocks + head
Strategy C – Full fine-tuning:    update all   parameters
Strategy D – LoRA adapters:       keep base    → add small trainable rank-r adapters
```

| Strategy | Trainable params | When to use | Risk |
|---|---|---|---|
| **Linear probe** | ~0.1% | Very small dataset (< 200 samples), sanity check | Under-fits if domain shift is large |
| **Partial fine-tuning** | 5–30% | Small dataset, moderate domain shift | Needs careful LR scheduling |
| **Full fine-tuning** | 100% | Large enough dataset (1000+ per class), high domain shift | Overfitting, high VRAM |
| **LoRA adapters** | 1–5% | Small dataset with large domain shift, low VRAM | Adds training complexity |

### Recommendation for Narrow Tasks

Start with a **linear probe** to set a fast baseline. Then use **LoRA or partial
fine-tuning** as the practical workhorse for narrow tasks. Full fine-tuning is rarely
worth the risk of overfitting until you have enough data.


In [ ]:
from transformers import ViTForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ── Strategy A: Linear probe ──────────────────────────────────────
model_linear = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CLASSES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
for name, param in model_linear.named_parameters():
    param.requires_grad = "classifier" in name

total, trainable = count_params(model_linear)
print(f"Linear probe  → total: {total:>10,}  trainable: {trainable:>8,}  "
      f"({trainable / total:.2%})")

# ── Strategy B: Partial fine-tuning (last 4 blocks + head) ────────
model_partial = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CLASSES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
for name, param in model_partial.named_parameters():
    param.requires_grad = False
for name, param in model_partial.named_parameters():
    if any(f"encoder.layer.{i}" in name for i in [8, 9, 10, 11]) or "classifier" in name or "layernorm" in name:
        param.requires_grad = True

total, trainable = count_params(model_partial)
print(f"Partial FT    → total: {total:>10,}  trainable: {trainable:>8,}  "
      f"({trainable / total:.2%})")

# ── Strategy D: LoRA adapters ──────────────────────────────────────
from peft import LoraConfig, get_peft_model

base_for_lora = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CLASSES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["query", "value"],
)
model_lora = get_peft_model(base_for_lora, lora_config)
total, trainable = count_params(model_lora)
print(f"LoRA (r=16)   → total: {total:>10,}  trainable: {trainable:>8,}  "
      f"({trainable / total:.2%})")

## 7. Training Setup

We use the LoRA strategy (Strategy D above) as the recommended approach for narrow tasks.

### Key Hyperparameter Decisions

| Parameter | Value used | Rationale |
|---|---|---|
| Learning rate | 3e-4 | Standard LoRA range; use warmup + cosine decay |
| Batch size | 16 | Fits comfortably in most 8 GB VRAM setups |
| Epochs | 20 | Small dataset needs more passes |
| Weighted loss | Yes | Balanced weight accounts for class frequency |
| Warmup | 10% of steps | Prevents early divergence |
| Weight decay | 0.01 | Regularisation to mitigate overfitting |

### Why Learning Rate Warmup Matters

ViT attention weights are sensitive early in fine-tuning. Jumping to a high LR in the
first steps can permanently disrupt representations. Warmup gives the heads time to
initialise before the full LR kicks in.


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader

BATCH_SIZE = 16
EPOCHS = 20
LR = 3e-4

train_ds = DefectDataset(train_df, transform=train_transform)
val_ds = DefectDataset(val_df, transform=val_transform)
test_ds = DefectDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps = EPOCHS * len(train_loader)
warmup_steps = int(0.10 * total_steps)

warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)
cosine_sched = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=LR * 0.01)
scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

print(f"Training samples:    {len(train_ds)}")
print(f"Validation samples:  {len(val_ds)}")
print(f"Test samples:        {len(test_ds)}")
print(f"Steps/epoch:         {len(train_loader)}")
print(f"Total steps:         {total_steps}")
print(f"Warmup steps:        {warmup_steps}")

## 8. Training Loop

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None, criterion=None, device="cpu"):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(pixel_values=imgs)
            logits = outputs.logits
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * len(labels)
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels


if RUN_TRAINING:
    model_lora = model_lora.to(device)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, _, _ = run_epoch(
            model_lora, train_loader, optimizer, scheduler, loss_fn, device
        )
        va_loss, va_acc, _, _ = run_epoch(
            model_lora, val_loader, criterion=loss_fn, device=device
        )
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.clone() for k, v in model_lora.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{EPOCHS}  "
                  f"train loss={tr_loss:.4f} acc={tr_acc:.3f}  "
                  f"val loss={va_loss:.4f} acc={va_acc:.3f}")

    model_lora.load_state_dict(best_state)
    model_save_path = ARTIFACT_DIR / "vit_lora_defect"
    model_lora.save_pretrained(str(model_save_path))
    print(f"Best val acc: {best_val_acc:.4f}")
    print(f"Adapter saved: {model_save_path}")
else:
    print("Training skipped. Set RUN_TRAINING = True to run.")

In [ ]:
if RUN_TRAINING:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    epochs_range = range(1, EPOCHS + 1)
    ax1.plot(epochs_range, history["train_loss"], label="Train")
    ax1.plot(epochs_range, history["val_loss"], label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Cross-Entropy Loss")
    ax1.legend()
    ax2.plot(epochs_range, history["train_acc"], label="Train")
    ax2.plot(epochs_range, history["val_acc"], label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.set_ylim([0, 1.05])
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Run training to see learning curves.")

## 9. Metrics That Matter for Narrow Tasks

### Why Accuracy Alone Is Misleading

On a balanced three-class problem, 90% accuracy sounds good.
On a defect detection problem where defects are 5% of images, a model that predicts
"no defect" every time achieves 95% accuracy — while completely missing every defect.

### The Right Metrics

| Metric | Formula | Narrow task use |
|---|---|---|
| **Per-class precision** | TP / (TP + FP) | How often a positive prediction is correct |
| **Per-class recall** | TP / (TP + FN) | How many actual defects are caught (sensitivity) |
| **Per-class F1** | 2 × P × R / (P + R) | Harmonic mean, penalises imbalanced P/R |
| **Macro F1** | Mean F1 across classes | Does not weight by class frequency |
| **Weighted F1** | F1 weighted by class support | Suitable when class frequency matters |
| **Confusion matrix** | Full error breakdown | Reveals which class is confused for which |
| **ROC AUC (per class)** | Area under TPR/FPR curve | Threshold-independent performance |

### Cost-Asymmetry

In defect detection, the two error types have very different costs:

| Error | Meaning | Typical cost |
|---|---|---|
| False negative | Missed defect passes inspection | High — product quality failure |
| False positive | Clean product flagged as defective | Low — unnecessary re-inspection |

Set the **decision threshold** based on the cost ratio, not at 0.5.


In [ ]:
# Simulate realistic predictions for demonstration
# (Replace with model_lora evaluation after training)
rng = np.random.default_rng(SEED)

test_labels_true = test_df["label"].map(label2id).tolist()

# Simulate a competent but imperfect model
def simulate_preds(true_labels, accuracy=0.82):
    preds = []
    for t in true_labels:
        if rng.random() < accuracy:
            preds.append(t)
        else:
            # bias errors: scratch confused with no_defect, crack confused with scratch
            confusion_map = {0: [1], 1: [0, 2], 2: [1]}
            preds.append(rng.choice(confusion_map[t]))
    return preds


if not RUN_INFERENCE_EVAL:
    test_preds = simulate_preds(test_labels_true, accuracy=0.82)
    print("Using simulated predictions for demonstration.")
    print("Set RUN_INFERENCE_EVAL = True to evaluate the actual trained model.")
else:
    model_lora = model_lora.to(device)
    _, _, test_preds, test_labels_true = run_epoch(
        model_lora, test_loader, criterion=loss_fn, device=device
    )
    print("Using actual model predictions.")

In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    test_labels_true, test_preds,
    target_names=CLASSES,
    digits=3,
))

macro_f1 = f1_score(test_labels_true, test_preds, average="macro")
weighted_f1 = f1_score(test_labels_true, test_preds, average="weighted")
accuracy = accuracy_score(test_labels_true, test_preds)

print(f"  Accuracy:     {accuracy:.3f}")
print(f"  Macro F1:     {macro_f1:.3f}")
print(f"  Weighted F1:  {weighted_f1:.3f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels_true, test_preds, labels=list(range(len(CLASSES))))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw counts
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0], linewidths=0.5)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title("Confusion Matrix (counts)")

# Normalised by true class (recall)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1], linewidths=0.5)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_title("Confusion Matrix (recall-normalised)")

plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision, recall, F1 bar chart
from sklearn.metrics import precision_recall_fscore_support

precs, recs, f1s, supports = precision_recall_fscore_support(
    test_labels_true, test_preds, labels=list(range(len(CLASSES)))
)

metric_df = pd.DataFrame({
    "Class": CLASSES,
    "Precision": precs,
    "Recall": recs,
    "F1": f1s,
    "Support": supports,
})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(CLASSES))
w = 0.25
bars = ax.bar(x - w, metric_df["Precision"], w, label="Precision")
bars2 = ax.bar(x, metric_df["Recall"], w, label="Recall")
bars3 = ax.bar(x + w, metric_df["F1"], w, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES)
ax.set_ylim([0, 1.1])
ax.set_ylabel("Score")
ax.set_title("Per-Class Precision / Recall / F1")
ax.legend()
ax.axhline(0.9, color="gray", linestyle="--", linewidth=0.8, label="0.9 target")
plt.tight_layout()
plt.show()

metric_df.set_index("Class")

## 10. Error Analysis

Metrics give you aggregate numbers. Error analysis tells you **which images fail and why**.

### Standard Error Analysis Workflow

1. Look at the most confidently wrong predictions (high probability assigned to the wrong class)
2. Look at the lowest-confidence correct predictions (the model barely got it right)
3. Slice by image source, acquisition condition, or time if metadata is available
4. Check whether errors cluster in one class — that often points to labelling issues or
   low-quality representatives of that class in training data


In [ ]:
# Simulate per-image softmax probabilities for the demonstration
def simulate_probas(true_labels, preds, accuracy=0.82):
    rng2 = np.random.default_rng(SEED + 1)
    n = len(true_labels)
    probas = np.zeros((n, len(CLASSES)))
    for i, (t, p) in enumerate(zip(true_labels, preds)):
        correct = (t == p)
        if correct:
            base = rng2.uniform(0.60, 0.98)
            probas[i, p] = base
            remaining = 1.0 - base
            others = [j for j in range(len(CLASSES)) if j != p]
            split = rng2.dirichlet(np.ones(len(others)))
            for j, o in enumerate(others):
                probas[i, o] = remaining * split[j]
        else:
            base = rng2.uniform(0.45, 0.75)
            probas[i, p] = base
            remaining = 1.0 - base
            others = [j for j in range(len(CLASSES)) if j != p]
            split = rng2.dirichlet(np.ones(len(others)))
            for j, o in enumerate(others):
                probas[i, o] = remaining * split[j]
    return probas


test_probas = simulate_probas(test_labels_true, test_preds)

error_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true": [id2label[t] for t in test_labels_true],
    "pred": [id2label[p] for p in test_preds],
    "confidence": [test_probas[i, p] for i, p in enumerate(test_preds)],
    "correct": [t == p for t, p in zip(test_labels_true, test_preds)],
})

wrong = error_df[~error_df["correct"]].sort_values("confidence", ascending=False)
print(f"Total test images:     {len(error_df)}")
print(f"Errors:                {len(wrong)}")
print(f"\nTop-5 most confident wrong predictions:")
print(wrong[["true", "pred", "confidence"]].head(5).to_string(index=False))

In [ ]:
# Plot confidently wrong examples
wrong_sample = wrong.head(min(6, len(wrong))).reset_index(drop=True)
n = len(wrong_sample)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3.5))
    if n == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, wrong_sample.iterrows()):
        img = Image.open(row["path"])
        ax.imshow(img)
        ax.set_title(
            f"True: {row['true']}\nPred: {row['pred']}\nConf: {row['confidence']:.2f}",
            fontsize=8,
            color="red",
        )
        ax.axis("off")
    plt.suptitle("Most Confident Incorrect Predictions", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No errors to display.")

## 11. Practical Constraints

### VRAM Budget

| Model | Resolution | Batch 16 | Batch 32 | Notes |
|---|---|---|---|---|
| ViT-Base (full FT) | 224 | ~14 GB | ~22 GB | Needs A100 or multi-GPU |
| ViT-Base (partial FT, last 4 blocks) | 224 | ~8 GB | ~12 GB | V100 / RTX 3090 |
| ViT-Base (LoRA r=16) | 224 | ~4 GB | ~7 GB | RTX 3060 / T4 |
| ViT-Small (LoRA r=16) | 224 | ~2 GB | ~3 GB | Consumer GPU |

**Gradient checkpointing** (`model.gradient_checkpointing_enable()`) can halve VRAM
at the cost of ~30% slower training.

### Input Resolution

| Resolution | Patch size | Sequence length | Notes |
|---|---|---|---|
| 224 | 16 | 196 tokens | Standard, tested, well-supported |
| 384 | 16 | 576 tokens | Better for fine detail; 3× more VRAM |
| 512 | 16 | 1024 tokens | High-res inspection; can exceed VRAM easily |

For defect detection, higher resolution almost always helps if VRAM allows.
Use 224 to iterate fast, then try 384 before production.

### Minimum Data Requirements

There is no universal rule, but rough empirical guidance for narrow classification:

| Quality of domain shift | Minimum train images per class |
|---|---|
| Mild (e.g., branded products on ImageNet-like backgrounds) | 50–100 |
| Moderate (e.g., industrial surface with good lighting) | 200–500 |
| Severe (e.g., medical histology, grayscale X-rays) | 500–2000 |

Below these thresholds, consider:
- data augmentation more aggressively
- using SimCLR / DINO-style self-supervised pre-training on unlabelled domain images
- manual review of the labelling quality before adding data

### Overfitting Warning Signs

Monitor these during training:

| Signal | Meaning | Response |
|---|---|---|
| Val loss rises while train loss drops | Classic overfitting | Increase dropout, reduce LR, add augmentation |
| Val accuracy plateaus early (< 5 epochs) | Learning rate too high or wrong strategy | Reduce LR or switch to linear probe |
| Train acc = 1.0 very early | Model memorising training set | Increase regularisation, reduce epoch count |
| Large per-class variance in F1 | Labelling issues or rare class starvation | Audit labels and augment minority class |


In [ ]:
# Memory and throughput estimation (demonstration — no actual GPU query)
import math

def estimate_vram_gb(seq_len, hidden_dim, num_layers, batch_size, bytes_per_param=2):
    # Rough activation memory per layer: batch * seq * hidden * 2 (forward + backward)
    activation_bytes = batch_size * seq_len * hidden_dim * 2 * num_layers * bytes_per_param
    # ViT-base parameters (~86M)
    param_bytes = 86e6 * bytes_per_param
    return (activation_bytes + param_bytes) / 1e9


configs = [
    ("ViT-B 224x224 batch=16", 196, 768, 12, 16),
    ("ViT-B 224x224 batch=32", 196, 768, 12, 32),
    ("ViT-B 384x384 batch=16", 576, 768, 12, 16),
    ("ViT-B 512x512 batch=8",  1024, 768, 12, 8),
]

print(f"{'Config':<35} {'Est. VRAM (GB)':>14}")
print("-" * 52)
for name, seq, hid, layers, bs in configs:
    vram = estimate_vram_gb(seq, hid, layers, bs)
    print(f"{name:<35} {vram:>13.1f}")
print("\nNote: these are rough estimates for educational illustration.")
print("Actual VRAM depends on optimizer states, gradient checkpointing, and framework overhead.")

In [ ]:
# Learning rate sensitivity illustration
lrs = [1e-5, 5e-5, 1e-4, 3e-4, 1e-3]
rng3 = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(9, 4))
epochs_x = np.arange(1, 21)

for lr in lrs:
    if lr < 1e-4:
        # underfit (too low LR)
        curve = 0.5 + (0.28 * np.log(epochs_x + 1) / np.log(22)) + rng3.normal(0, 0.01, 20)
    elif lr <= 3e-4:
        # sweet spot
        curve = 0.55 + 0.33 * (1 - np.exp(-epochs_x / 7)) + rng3.normal(0, 0.01, 20)
    else:
        # diverges
        noise = rng3.normal(0, 0.04, 20)
        curve = 0.55 + 0.2 * (1 - np.exp(-epochs_x / 3)) + noise
        curve[8:] = curve[8:] - np.linspace(0, 0.25, 12)  # late degradation

    curve = np.clip(curve, 0.3, 1.0)
    ax.plot(epochs_x, curve, label=f"LR = {lr:.0e}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Simulated Val Accuracy")
ax.set_title("LR Sensitivity on a Narrow Fine-Tuning Task (illustrative)")
ax.legend(fontsize=9)
ax.set_ylim([0.2, 1.0])
plt.tight_layout()
plt.show()

## 12. Save Experiment Log

In [ ]:
from datetime import datetime

log = {
    "timestamp": datetime.now().isoformat(),
    "task": "vision_model_fine_tuning",
    "dataset": "synthetic_surface_defects",
    "classes": CLASSES,
    "n_per_class": N_PER_CLASS,
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
    "base_model": BASE_MODEL,
    "strategy": "LoRA_r16_query_value",
    "simulated_test_accuracy": float(accuracy_score(test_labels_true, test_preds)),
    "simulated_macro_f1": float(f1_score(test_labels_true, test_preds, average="macro")),
}

log_path = ARTIFACT_DIR / "vision_finetune_log.json"
log_path.write_text(json.dumps(log, indent=2), encoding="utf-8")
print(f"Saved: {log_path}")

## 13. What to Do When It Isn't Working

| Symptom | Most likely cause | Fix |
|---|---|---|
| Accuracy barely above chance | Domain too far from pre-training | Use SSL pre-training on unlabelled domain images first |
| Overfits in 3–5 epochs | Not enough training data | More aggressive augmentation, lower LR, fewer trainable params |
| One class consistently bad | Class underrepresented or mislabelled | Audit that class's labels, add targeted augmentation |
| Loss NaN | LR too high | Reduce by 10×, add gradient clipping |
| Prediction always one class | Severe imbalance, loss not weighted | Add class weights or focal loss, check data loading |
| VRAM error | Batch size too large for model+resolution | Reduce batch and use gradient accumulation |

### Quick Decision Tree

```
Do you have fewer than 200 images per class?
  └─ YES → aggressive augmentation + linear probe baseline first
       └─ Domain shift severe? → SSL pre-training on unlabelled data
  └─ NO → LoRA (default) or partial fine-tuning

Does recall on the minority class matter more than precision?
  └─ YES → weighted loss + set threshold < 0.5 for the minority class
  └─ NO → standard training

Tight VRAM budget?
  └─ YES → LoRA, gradient checkpointing, batch=8 + accumulate 4 steps
  └─ NO → partial fine-tuning or full fine-tuning
```


## 14. Key Takeaways

### Dataset Preparation
- Always stratify train/val/test splits by class label
- Use different augmentation pipelines for training vs. inference
- Compute class weights before training — do not assume balance
- For narrow tasks, augmentation is not optional; it is the primary defence against overfitting

### Model Selection
- A linear probe establishes a fast baseline and confirms the pre-training features are useful
- LoRA with `r=16` on query/value projections is the practical default for narrow tasks
- Only move to full fine-tuning when you have 500+ images per class and enough VRAM

### Evaluation
- Never report only accuracy for narrow tasks — add per-class precision, recall, and F1
- Use a confusion matrix to identify which pairs of classes are confused
- Tune the decision threshold to match the real cost ratio (false negative vs. false positive)
- Error analysis on most-confident wrong predictions reveals systematic labelling or data issues

### Practical Constraints
- VRAM is the first bottleneck; gradient checkpointing cuts it roughly in half
- Higher resolution always helps defect detail, but triples sequence length and VRAM
- Warmup prevents early divergence; cosine decay prevents late-stage oscillation
- Below ~200 images per class, consider self-supervised domain pre-training before supervised fine-tuning
